# Scrape & Process RaceResult events

Two clearly separated stages (see `race_results/parsers/raceresult_common.py`):

1. **Scrape** — pull each event's lists from the my.raceresult.com JSON API and dump them to *raw* CSVs in `race_results/raw/` with **no interpretation** (times as strings, `Last, First` names, place keeps its dot).
2. **Process** — turn the raw CSVs into the standard 13-column *processed* CSVs in `race_results/` that the app loads.

Both stages **overwrite** their output files, so re-running never duplicates rows. The app's metric columns are computed at load time by `metrics.build_metrics_from_df`, so they are never baked into the CSV.

**To add a race:** add an `EventSpec` to `race_results/parsers/raceresult_events.py` and re-run this notebook. `races.py` builds its `REGISTRY` straight from that list, so the app picks it up automatically — no other edits, and a name can never be registered twice.

In [ ]:
from race_results.parsers.raceresult_events import EVENTS, scrape_all, process_all, build_all
from race_results.parsers.raceresult_common import scrape_url, load_processed_csv

for spec in EVENTS:
    print(f"{spec.name:24s} event={spec.event_id}  ->  {spec.processed_filename}")

## 1. Scrape — raw CSVs (no processing)

In [ ]:
scrape_all()

import os
for f in sorted(os.listdir('race_results/raw')):
    print('raw/' + f)

## 2. Process — standard 13-column processed CSVs

In [ ]:
paths = process_all()
for name, p in paths.items():
    df = load_processed_csv(p)
    print(f"{name:24s} rows={len(df):4d}  ->  {p.name}")

## 3. Verify the app sees the results

`races.load_all` reads the processed CSVs through the registry and adds the metric columns. A clean run here means the Dash app (`app.py`) will load the same data.

In [ ]:
from races import REGISTRY, load_all

print('REGISTRY:')
for name, (path, loader) in REGISTRY.items():
    print(f'  {name:24s} {path}')

print('\nload_all (with metrics):')
dfs = load_all()
for name, df in dfs.items():
    print(f'  {name:24s} rows={len(df):4d}  cols={df.shape[1]}')

## General scraper for any RaceResult URL

`scrape_url` dumps raw CSVs for any my.raceresult.com results link (or bare event id). With no `listnames` it discovers and dumps every list published on the results page — handy for inspecting a new event's `DataFields` before writing its `EventSpec`.

In [ ]:
# Example: discover and dump every list for a new event, then inspect a list's columns.
# written = scrape_url('https://my.raceresult.com/403737/')
# import pandas as pd
# raw = pd.read_csv(next(iter(written.values())), dtype=str, keep_default_na=False)
# print(list(raw.columns))   # the API DataFields -> column indices for a new EventSpec
# raw.head()